In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor

!pwd

!ls

/content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor
/content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor
data  notebooks  outputs  requirements.txt  src


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 126.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.27.0
    Uninstalling huggingface_hub-1.27.0:
      Successfully uninstalled huggingface_hub-1.27.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninst

In [4]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [5]:
# DATASET_NAME = "databricks/databricks-dolly-15k"

path_files = {
        "train" : "data/raw/train.jsonl",
        "validate" : "data/raw/validate.jsonl",
        "test" : "data/raw/test.jsonl"
    }
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [ ]:
from src.tokenizer.load_tokenizer import load_tokenizer

tokenizer = load_tokenizer(model_name=MODEL_NAME)

In [6]:
from src.data.pipeline import pipeline

train, validate, test = pipeline(path_files)
print(train)
print(validate)
print(test)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validate split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 1600
})
Dataset({
    features: ['messages'],
    num_rows: 200
})
Dataset({
    features: ['messages'],
    num_rows: 200
})


In [ ]:
from src.model.pipeline import pipeline

base_model, training_model = pipeline(MODEL_NAME)

In [ ]:
from src.train.pipeline import pipeline

pipeline(dataset={"train":train, "validate":validate}, model=training_model, tokenizer=tokenizer)

In [7]:
from src.inference.pipeline import pipeline

base_model, finetune_model, finetuned_tokenizer = pipeline()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

adapter_model.safetensors:   0%|          | 0.00/9.03M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

In [ ]:
from src.inference.generate_response import response

import os
import json

def save_res(id, question, base_model_res, finetuned_model_res):
    result = {
        "id": id,
        "question": question,
        "base_model": base_model_res,
        "finetuned_model": finetuned_model_res
    }

    os.makedirs("evaluation", exist_ok=True)

    with open("evaluation/result.jsonl", "a", encoding="utf-8") as f:
        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            ) + "\n"
        )

for i in range(len(test)):
    question = [test[i]['messages'][0]]

    base_model_res = response(model=base_model, tokenizer=finetuned_tokenizer, message=question)
    finetuned_model_res = response(model=finetune_model, tokenizer=finetuned_tokenizer, message=question)

    save_res(i+1, question=question[0]['content'], base_model_res=base_model_res, finetuned_model_res=finetuned_model_res)

    print(f"{question[0]['content']=}")
    print(f"{base_model_res=}")
    print(f"{finetuned_model_res=}\n")

200
